please change the paths to the adapters accordingly

In [1]:
# Install Unsloth and dependencies
!pip install -q --upgrade pip

!pip install -q \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install -q \
    transformers \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import re
import torch
import pandas as pd

from datasets import Dataset

from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from peft import PeftModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
MAX_SEQ_LENGTH = 2048
MAX_PROMPT_LENGTH = 2048
MAX_COMPLETION_LENGTH = 32

NUM_EPOCHS = 1
LEARNING_RATE = 5e-6

PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

NUM_GENERATIONS = 4

## Instructions

Before running this notebook, update the configuration variables below.

### 1. Select the GRPO Reward Model

Set `MODEL_NUM` according to the reward function you want to train:

```python
# MODEL_NUM = 1, 2, or 3
MODEL_NUM = 1
```

| `MODEL_NUM` | Reward Model |
|-------------|--------------|
| **1** | Exact Ranking Reward |
| **2** | Relative Ranking Reward |
| **3** | Composite Ranking Reward |

---

### 2. Update the SFT Adapter Path

Set `SFT_ADAPTER_PATH` to the directory containing your previously trained SFT adapter.

```python
SFT_ADAPTER_PATH = "/path/to/sft_adapter"
```

This can either be:

- The SFT adapter generated by the provided `sft_training.ipynb`, or
- An existing SFT adapter stored on your local machine or Google Drive.

---

### 3. Update the GRPO Output Directory

Specify the directory where the trained GRPO adapter will be saved.

```python
OUTPUT_DIR = "/path/to/grpo_model_adapter"
```

For example:

```python
OUTPUT_DIR = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/grpo_model3_adapter"
```


In [ ]:
import re
import torch
import pandas as pd
from datasets import Dataset

from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOTrainer, GRPOConfig

#MODEL_NUM = 1 OR 2 OR 3
MODEL_NUM = 3

BASE_MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

TRAIN_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train_subset.csv"

OUTPUT_DIR = f"/content/drive/MyDrive/colm_grpo_llmaj_dataset/grpo_model{MODEL_NUM}_adapter"
SFT_ADAPTER_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/sft_adapter"


In [ ]:
VALID_RANKINGS = [
    "R1>R2>R3",
    "R1>R3>R2",
    "R2>R1>R3",
    "R2>R3>R1",
    "R3>R1>R2",
    "R3>R2>R1",
]

RANKING_RE = re.compile(r"R[123]\s*>\s*R[123]\s*>\s*R[123]")


def normalize_ranking(text):
    if text is None:
        return None

    match = RANKING_RE.search(str(text).strip())
    if not match:
        return None

    ranking = match.group(0).replace(" ", "")
    return ranking if ranking in VALID_RANKINGS else None


def get_text(completion):
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        if len(completion) > 0 and isinstance(completion[0], dict):
            return completion[0].get("content", "")
        return str(completion[0])

    return str(completion)


def format_reward(pred):
    return 0.5 if pred is not None else 0.0


def ranking_to_pairs(ranking):
    if ranking is None:
        return set()

    items = ranking.split(">")
    pairs = set()

    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            pairs.add((items[i], items[j]))

    return pairs


def pairwise_agreement(pred, gold):
    pred_pairs = ranking_to_pairs(pred)
    gold_pairs = ranking_to_pairs(gold)

    if len(pred_pairs) != 3 or len(gold_pairs) != 3:
        return 0.0

    return len(pred_pairs.intersection(gold_pairs)) / 3.0

def make_prompt(row):
    return f"""You are an expert cybersecurity answer evaluator.

You will be given a cybersecurity question and three candidate answers.

Your task is to rank the three answers from best to worst based on:
1. Technical correctness
2. Completeness
3. Relevance to the question
4. Clarity and precision
5. Lack of hallucination or misleading information
6. Keywords match

Assign each rubric a score and then sum all 6 rubric scores to find total score for a response and then rank the responses.

Important:
- R1, R2, and R3 are all candidate answers.
- Do not assume the reference answer is always best.
- Judge only based on answer quality.
- Output ONLY the ranking.
- The output format must be exactly like one of these:
R1>R2>R3
R1>R3>R2
R2>R1>R3
R2>R3>R1
R3>R1>R2
R3>R2>R1

Question:
{row["question"]}

R1:
{row["answer"]}

R2:
{row["candidate_llama_3_2_1b_instruct"]}

R3:
{row["candidate_qwen3_32b"]}

Ranking:"""


df = pd.read_csv(TRAIN_CSV)

required_cols = [
    "question",
    "answer",
    "candidate_llama_3_2_1b_instruct",
    "candidate_qwen3_32b",
    "gpt_51_judge_ranking",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna(subset=required_cols).reset_index(drop=True)

df["prompt"] = df.apply(make_prompt, axis=1)
df["gold_ranking"] = df["gpt_51_judge_ranking"].apply(normalize_ranking)

df = df.dropna(subset=["gold_ranking"]).reset_index(drop=True)

train_dataset = Dataset.from_pandas(df[["prompt", "gold_ranking"]])



model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(
    model,
    SFT_ADAPTER_PATH,
    is_trainable=True,
)

FastLanguageModel.for_training(model)

print("Loaded SFT adapter for GRPO:", SFT_ADAPTER_PATH)


def exact_ranking_reward(completions, gold_ranking, **kwargs):
    rewards = []

    for completion, gold in zip(completions, gold_ranking):
        pred = normalize_ranking(get_text(completion))

        correctness = 1.0 if pred == gold else -1.0
        fmt = format_reward(pred)

        rewards.append(correctness + fmt)

    return rewards


def relative_ranking_reward(completions, gold_ranking, **kwargs):
    rewards = []

    for completion, gold in zip(completions, gold_ranking):
        pred = normalize_ranking(get_text(completion))

        if pred is None:
            pairwise = -1.0
        else:
            score = pairwise_agreement(pred, gold)
            pairwise = (2 * score) - 1

        fmt = format_reward(pred)
        rewards.append(pairwise + fmt)

    return rewards


def composite_ranking_reward(completions, gold_ranking, **kwargs):
    rewards = []
    weights = [0.5, 0.3, 0.2]

    for completion, gold in zip(completions, gold_ranking):
        pred = normalize_ranking(get_text(completion))

        if pred is None:
            ranking_reward = -1.0
        else:
            pred_items = pred.split(">")
            gold_items = gold.split(">")

            s_rank = sum(
                w for w, p, g in zip(weights, pred_items, gold_items)
                if p == g
            )

            ranking_reward = (2 * s_rank) - 1

        fmt = format_reward(pred)
        rewards.append(ranking_reward + fmt)

    return rewards


if MODEL_NUM == 1:
    reward_funcs = [exact_ranking_reward]
elif MODEL_NUM == 2:
    reward_funcs = [relative_ranking_reward]
elif MODEL_NUM == 3:
    reward_funcs = [composite_ranking_reward]
else:
    raise ValueError("MODEL_NUM must be 1, 2, or 3")


training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,

    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,

    fp16=False,
    bf16=True,

    logging_steps=10,
    save_steps=25,
    save_total_limit=2,
    temperature=0.7,
    top_p=0.9,

    report_to="none",
)


trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()


model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved GRPO Model {MODEL_NUM} adapter to:", OUTPUT_DIR)